# Build Water Entity Dictionary

Builds `output/water_entity_dictionary.csv` — CA water governance entities for use as a spaCy entity ruler dictionary in the IRWM NER pipeline.

Each row has:
- `all_names`: pipe-separated list of the full name + common abbreviations/aliases (the R pipeline splits on `|`)
- `entity_type`: category for reference/filtering
- `notes`: optional provenance or disambiguation notes

**Sources / update guidance:**
- CA state agencies: https://www.ca.gov/agencies/
- SWRCB regional boards: https://www.waterboards.ca.gov/
- IRWM regions / lead agencies: https://water.ca.gov/Programs/Integrated-Regional-Water-Management
- Federal agencies: agency websites
- Water districts: ACWA membership, DWR water supplier list
- Tribes: CWSRF tribal list, BIA CA tribes
- NGOs: project acronym/glossary sections (supplemented by regional JSONs)

In [1]:
import os
import pandas as pd

OUTPUT_DIR = "../../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "water_entity_dictionary.csv")

## CA State Agencies

In [2]:
state_agencies = [
    # Primary water agencies
    ("Department of Water Resources|DWR|California Department of Water Resources|California DWR", "state_agency", ""),
    ("State Water Resources Control Board|SWRCB|State Water Board|California State Water Board", "state_agency", ""),
    ("California Department of Fish and Wildlife|CDFW|Department of Fish and Wildlife", "state_agency", "formerly CDFG"),
    ("California Department of Fish and Game|CDFG|Department of Fish and Game", "state_agency", "predecessor to CDFW"),
    ("California Environmental Protection Agency|CalEPA|Cal EPA", "state_agency", ""),
    ("Department of Toxic Substances Control|DTSC", "state_agency", ""),
    ("California Department of Public Health|CDPH|Department of Public Health", "state_agency", ""),
    ("Office of Environmental Health Hazard Assessment|OEHHA", "state_agency", ""),
    ("California Air Resources Board|CARB|Air Resources Board", "state_agency", ""),
    ("Governor's Office of Planning and Research|OPR|Office of Planning and Research", "state_agency", ""),
    ("California Natural Resources Agency|CNRA|Natural Resources Agency", "state_agency", ""),
    ("California Department of Food and Agriculture|CDFA|Department of Food and Agriculture", "state_agency", ""),
    ("California Office of Emergency Services|Cal OES|OES", "state_agency", ""),
    ("California Department of Conservation|DOC|Department of Conservation", "state_agency", ""),
    ("California Department of Forestry and Fire Protection|CAL FIRE|CalFire|CDFFP", "state_agency", ""),
    ("California Coastal Commission|CCC|Coastal Commission", "state_agency", ""),
    ("California Department of Transportation|Caltrans|CDOT", "state_agency", ""),
    ("State Lands Commission|California State Lands Commission|SLC", "state_agency", ""),
    ("California Energy Commission|CEC|Energy Commission", "state_agency", ""),
    ("Public Utilities Commission|California Public Utilities Commission|CPUC", "state_agency", ""),
    ("Delta Stewardship Council|DSC", "state_agency", ""),
    ("Delta Watermaster", "state_agency", ""),
    ("Sacramento-San Joaquin Delta Conservancy|Delta Conservancy", "state_agency", ""),
    ("California Wildlife Conservation Board|WCB|Wildlife Conservation Board", "state_agency", ""),
    ("California State Coastal Conservancy|Coastal Conservancy|SCC", "state_agency", ""),
    ("Baldwin Hills Conservancy", "state_agency", ""),
    ("Santa Monica Mountains Conservancy|SMMC", "state_agency", ""),
    ("San Gabriel and Lower Los Angeles Rivers and Mountains Conservancy|RMC", "state_agency", ""),
    ("San Joaquin River Conservancy", "state_agency", ""),
    ("Coachella Valley Mountains Conservancy|CVMC", "state_agency", ""),
    ("Sierra Nevada Conservancy|SNC", "state_agency", ""),
    ("California Tahoe Conservancy|CTC", "state_agency", ""),
    ("California Department of Water Resources Flood Operations Center", "state_agency", ""),
    ("Division of Safety of Dams|DSOD", "state_agency", "within DWR"),
    ("California Water Commission|CWC", "state_agency", ""),
    ("State Board of Forestry and Fire Protection", "state_agency", ""),
    ("California Department of Housing and Community Development|HCD", "state_agency", ""),
    ("Infrastructure and Economic Development Bank|IBank|California IBank", "state_agency", ""),
]

## SWRCB Regional Water Quality Control Boards

In [3]:
regional_boards = [
    ("North Coast Regional Water Quality Control Board|North Coast Regional Board|Region 1 Water Board", "regional_water_board", "Region 1"),
    ("San Francisco Bay Regional Water Quality Control Board|San Francisco Bay Regional Board|Region 2 Water Board", "regional_water_board", "Region 2"),
    ("Central Coast Regional Water Quality Control Board|Central Coast Regional Board|Region 3 Water Board", "regional_water_board", "Region 3"),
    ("Los Angeles Regional Water Quality Control Board|Los Angeles Regional Board|Region 4 Water Board", "regional_water_board", "Region 4"),
    ("Central Valley Regional Water Quality Control Board|Central Valley Regional Board|Region 5 Water Board", "regional_water_board", "Region 5"),
    ("Lahontan Regional Water Quality Control Board|Lahontan Regional Board|Region 6 Water Board", "regional_water_board", "Region 6"),
    ("Colorado River Basin Regional Water Quality Control Board|Colorado River Basin Regional Board|Region 7 Water Board", "regional_water_board", "Region 7"),
    ("Santa Ana Regional Water Quality Control Board|Santa Ana Regional Board|Region 8 Water Board", "regional_water_board", "Region 8"),
    ("San Diego Regional Water Quality Control Board|San Diego Regional Board|Region 9 Water Board", "regional_water_board", "Region 9"),
]

## Federal Agencies

In [4]:
federal_agencies = [
    ("Bureau of Reclamation|USBR|Bureau of Reclamation Mid-Pacific Region|Mid-Pacific Region|US Bureau of Reclamation", "federal_agency", ""),
    ("US Army Corps of Engineers|Army Corps of Engineers|USACE|Corps of Engineers", "federal_agency", ""),
    ("US Environmental Protection Agency|US EPA|EPA Region 9|Environmental Protection Agency", "federal_agency", ""),
    ("US Geological Survey|USGS", "federal_agency", ""),
    ("US Fish and Wildlife Service|USFWS|Fish and Wildlife Service", "federal_agency", ""),
    ("National Marine Fisheries Service|NMFS|NOAA Fisheries", "federal_agency", ""),
    ("Natural Resources Conservation Service|NRCS", "federal_agency", ""),
    ("US Forest Service|USFS|Forest Service", "federal_agency", ""),
    ("Bureau of Land Management|BLM", "federal_agency", ""),
    ("National Park Service|NPS", "federal_agency", ""),
    ("Federal Emergency Management Agency|FEMA", "federal_agency", ""),
    ("US Department of Agriculture|USDA", "federal_agency", ""),
    ("US Department of the Interior|DOI|Department of the Interior", "federal_agency", ""),
    ("US Department of Housing and Urban Development|HUD", "federal_agency", ""),
    ("Western Area Power Administration|WAPA", "federal_agency", ""),
    ("Bureau of Indian Affairs|BIA", "federal_agency", ""),
    ("Environmental Protection Agency Office of Water", "federal_agency", ""),
    ("Farm Service Agency|FSA", "federal_agency", ""),
    ("Rural Utilities Service|RUS", "federal_agency", ""),
]

## Water Districts, Utilities, and Joint Powers Authorities

In [5]:
water_districts = [
    # Southern California
    ("Metropolitan Water District of Southern California|Metropolitan Water District|MWD|MET", "water_district", "largest urban water agency in US"),
    ("Los Angeles Department of Water and Power|LADWP|LA DWP", "water_district", ""),
    ("Los Angeles County Department of Public Works|LACDPW|LA County DPW", "water_district", ""),
    ("Los Angeles County Flood Control District|LACFCD", "water_district", ""),
    ("Inland Empire Utilities Agency|IEUA", "water_district", ""),
    ("Three Valleys Municipal Water District|Three Valleys MWD", "water_district", ""),
    ("Upper San Gabriel Valley Municipal Water District|Upper San Gabriel MWD", "water_district", ""),
    ("Central Basin Municipal Water District|Central Basin MWD", "water_district", ""),
    ("West Basin Municipal Water District|West Basin MWD", "water_district", ""),
    ("Municipal Water District of Orange County|MWDOC", "water_district", ""),
    ("San Diego County Water Authority|SDCWA|SD County Water Authority", "water_district", ""),
    ("Coachella Valley Water District|CVWD", "water_district", ""),
    ("Desert Water Agency|DWA", "water_district", ""),
    ("Imperial Irrigation District|IID", "water_district", ""),
    ("Ventura County Watershed Protection District|VCWPD", "water_district", ""),
    ("Calleguas Municipal Water District|Calleguas MWD", "water_district", ""),
    ("Antelope Valley-East Kern Water Agency|AVEK", "water_district", ""),
    ("Mojave Water Agency|MWA", "water_district", ""),
    ("Santa Ana Watershed Project Authority|SAWPA", "water_district", ""),
    ("Orange County Water District|OCWD", "water_district", ""),
    ("Orange County Sanitation District|OCSD", "water_district", ""),
    # Bay Area
    ("East Bay Municipal Utility District|EBMUD", "water_district", ""),
    ("San Francisco Public Utilities Commission|SFPUC|SF PUC", "water_district", ""),
    ("Santa Clara Valley Water District|Valley Water|SCVWD", "water_district", "formerly SCVWD"),
    ("Zone 7 Water Agency|Zone 7", "water_district", "Livermore-Amador Valley"),
    ("Contra Costa Water District|CCWD", "water_district", ""),
    ("Marin Municipal Water District|MMWD", "water_district", ""),
    ("North Marin Water District|NMWD", "water_district", ""),
    ("Sonoma County Water Agency|SCWA|Sonoma Water", "water_district", ""),
    ("Napa County Flood Control and Water Conservation District", "water_district", ""),
    ("Alameda County Water District|ACWD", "water_district", ""),
    ("San Jose Water Company|SJW", "water_district", ""),
    # Central Valley
    ("Fresno Metropolitan Flood Control District|Fresno MFCD", "water_district", ""),
    ("Kings River Conservation District|KRCD", "water_district", ""),
    ("Kern County Water Agency|KCWA", "water_district", ""),
    ("Tulare Lake Basin Water Storage District", "water_district", ""),
    ("Friant Water Authority|FWA", "water_district", ""),
    ("Westlands Water District|WWD", "water_district", "largest agricultural water district in US"),
    ("Merced Irrigation District|MID", "water_district", ""),
    ("Turlock Irrigation District|TID", "water_district", ""),
    ("Modesto Irrigation District|MID", "water_district", ""),
    ("San Luis Delta-Mendota Water Authority|SLDMWA", "water_district", ""),
    ("Glenn-Colusa Irrigation District|GCID", "water_district", ""),
    ("Sacramento Municipal Utility District|SMUD", "water_district", "power but relevant to water"),
    ("Sacramento Area Flood Control Agency|SAFCA", "water_district", ""),
    ("Sacramento County Water Agency|SCWA", "water_district", ""),
    ("El Dorado Irrigation District|EID", "water_district", ""),
    ("Placer County Water Agency|PCWA", "water_district", ""),
    ("Nevada Irrigation District|NID", "water_district", ""),
    ("Yuba County Water Agency|YCWA", "water_district", ""),
    ("Colusa County Water District", "water_district", ""),
    ("Tehama-Colusa Canal Authority|TCCA", "water_district", ""),
    # Central Coast
    ("Monterey Peninsula Water Management District|MPWMD", "water_district", ""),
    ("Monterey County Water Resources Agency|MCWRA", "water_district", ""),
    ("Pajaro Valley Water Management Agency|PVWMA", "water_district", ""),
    ("Santa Barbara County Water Agency|SBCWA", "water_district", ""),
    ("Cachuma Operation and Maintenance Board|COMB", "water_district", ""),
    ("San Luis Obispo County Flood Control and Water Conservation District", "water_district", ""),
    # North Coast / Mountains
    ("Humboldt Bay Municipal Water District|HBMWD", "water_district", ""),
    ("Shasta County Water Agency", "water_district", ""),
    ("Tehama County Flood Control and Water Conservation District", "water_district", ""),
    ("Truckee Meadows Water Authority|TMWA", "water_district", "Nevada but relevant to CA water"),
    ("Tahoe Regional Planning Agency|TRPA", "water_district", "bi-state"),
]

## IRWM Regional Collaboratives and Lead Agencies

In [6]:
irwm_entities = [
    ("Integrated Regional Water Management|IRWM|Integrated Regional Water Management Program", "irwm_program", ""),
    ("IRWM Plan|Integrated Regional Water Management Plan|IRWMP", "irwm_program", ""),
    ("Proposition 84|Prop 84", "irwm_program", "Safe Drinking Water, Water Quality and Supply, Flood Control, River and Coastal Protection Bond Act of 2006"),
    ("Proposition 1|Prop 1", "irwm_program", "Water Quality, Supply, and Infrastructure Improvement Act of 2014"),
    ("Proposition 50|Prop 50", "irwm_program", "Water Security, Clean Drinking Water, Coastal and Beach Protection Act of 2002"),
    ("Proposition 13|Prop 13", "irwm_program", "Safe Drinking Water, Clean Water, Watershed Protection, and Flood Protection Act of 2000"),
    # Region leads (selected key ones)
    ("North Coast Integrated Regional Water Management|North Coast IRWM", "irwm_region", "Region 1"),
    ("Sacramento River Integrated Regional Water Management|Sacramento River IRWM", "irwm_region", ""),
    ("Greater Los Angeles County IRWM|GLAC IRWM|Greater Los Angeles County Integrated Regional Water Management", "irwm_region", "Region 10"),
    ("San Francisco Bay Area Integrated Regional Water Management|Bay Area IRWM", "irwm_region", ""),
    ("Santa Ana Watershed Integrated Regional Water Management|Santa Ana IRWM", "irwm_region", ""),
    ("San Diego Integrated Regional Water Management|San Diego IRWM", "irwm_region", ""),
    ("Westside Sacramento Integrated Regional Water Management|Westside Sacramento IRWM", "irwm_region", "Region 45"),
    ("Yolo County Flood Control and Water Conservation District|Yolo County FCWCD", "water_district", ""),
]

## Tribes

In [7]:
tribes = [
    ("Hoopa Valley Tribe|Hoopa Valley Indian Tribe", "tribe", ""),
    ("Yurok Tribe|Yurok Indian Tribe", "tribe", ""),
    ("Karuk Tribe|Karuk Indian Tribe", "tribe", ""),
    ("Klamath Tribes", "tribe", ""),
    ("Round Valley Indian Tribes", "tribe", ""),
    ("Pomo Tribes|Pomo Indians", "tribe", ""),
    ("Dry Creek Rancheria Band of Pomo Indians", "tribe", ""),
    ("Federated Indians of Graton Rancheria", "tribe", ""),
    ("Wiyot Tribe", "tribe", ""),
    ("Shasta Indian Nation", "tribe", ""),
    ("Pit River Tribe", "tribe", ""),
    ("Yana Tribe", "tribe", ""),
    ("Winnemem Wintu Tribe", "tribe", ""),
    ("Mechoopda Indian Tribe", "tribe", ""),
    ("United Auburn Indian Community", "tribe", ""),
    ("Wilton Rancheria", "tribe", ""),
    ("Ione Band of Miwok Indians", "tribe", ""),
    ("Tuolumne Band of Me-Wuk Indians|Tuolumne Me-Wuk", "tribe", ""),
    ("Mooretown Rancheria of Maidu Indians", "tribe", ""),
    ("Enterprise Rancheria of Maidu Indians", "tribe", ""),
    ("Tule River Indian Tribe", "tribe", ""),
    ("Yokuts Tribe|Yokuts People", "tribe", ""),
    ("Mono Tribe|North Fork Rancheria of Mono Indians", "tribe", ""),
    ("Bishop Paiute Tribe", "tribe", ""),
    ("Fort Independence Indian Community", "tribe", ""),
    ("Big Pine Paiute Tribe of the Owens Valley", "tribe", ""),
    ("Agua Caliente Band of Cahuilla Indians", "tribe", ""),
    ("Cahuilla Band of Mission Indians", "tribe", ""),
    ("Torres Martinez Desert Cahuilla Indians", "tribe", ""),
    ("Soboba Band of Luiseno Indians", "tribe", ""),
    ("Rincon Band of Luiseno Mission Indians", "tribe", ""),
    ("Pala Band of Mission Indians", "tribe", ""),
    ("San Manuel Band of Mission Indians", "tribe", ""),
    ("Morongo Band of Mission Indians", "tribe", ""),
    ("Quechan Tribe|Fort Yuma-Quechan Tribe", "tribe", ""),
    ("Kumeyaay Nation|Kumeyaay People", "tribe", ""),
    ("Chumash People|Santa Ynez Band of Chumash Mission Indians", "tribe", ""),
    ("Ohlone People|Ohlone Tribe", "tribe", ""),
    ("Miwok Tribe|Coast Miwok", "tribe", ""),
    ("Wintun People|Wintun Nation", "tribe", ""),
]

## NGOs, Foundations, and Advocacy Organizations

In [8]:
ngos = [
    ("The Nature Conservancy|TNC|Nature Conservancy", "ngo", ""),
    ("National Audubon Society|Audubon Society|Audubon California", "ngo", ""),
    ("Sierra Club|Sierra Club California", "ngo", ""),
    ("Environmental Defense Fund|EDF", "ngo", ""),
    ("Natural Resources Defense Council|NRDC", "ngo", ""),
    ("Trout Unlimited", "ngo", ""),
    ("Ducks Unlimited", "ngo", ""),
    ("California Trout|CalTrout", "ngo", ""),
    ("California Waterfowl Association|CWA", "ngo", ""),
    ("Pacific Coast Federation of Fishermen's Associations|PCFFA", "ngo", ""),
    ("Association of California Water Agencies|ACWA", "ngo", "industry association"),
    ("California Farm Bureau Federation|CFBF|Farm Bureau", "ngo", ""),
    ("California Cattlemen's Association", "ngo", ""),
    ("California Rice Commission", "ngo", ""),
    ("California Grape and Tree Fruit League", "ngo", ""),
    ("State Water Contractors|SWC", "ngo", "State Water Project contractors group"),
    ("Central Valley Project Water Association|CVPWA", "ngo", ""),
    ("Sustainable Conservation", "ngo", ""),
    ("Pacific Institute", "ngo", ""),
    ("Water Education Foundation|WEF", "ngo", ""),
    ("Resource Conservation District|RCD", "ngo", "generic; specific RCDs named in regional dicts"),
    ("Resource Conservation Districts of California", "ngo", ""),
    ("American Rivers", "ngo", ""),
    ("Heal the Bay", "ngo", ""),
    ("Surfrider Foundation", "ngo", ""),
    ("TreePeople", "ngo", ""),
    ("San Francisco Estuary Institute|SFEI", "ngo", ""),
    ("Point Blue Conservation Science|Point Blue", "ngo", "formerly PRBO"),
    ("Sonoma Ecology Center", "ngo", ""),
    ("Napa County Resource Conservation District|Napa RCD", "ngo", ""),
]

## Compile and Write CSV

In [9]:
all_entries = (
    state_agencies
    + regional_boards
    + federal_agencies
    + water_districts
    + irwm_entities
    + tribes
    + ngos
)

df = pd.DataFrame(all_entries, columns=["all_names", "entity_type", "notes"])

# Sanity check: no empty all_names, no leading/trailing pipes
assert df["all_names"].str.startswith("|").sum() == 0, "Leading pipe found"
assert df["all_names"].str.endswith("|").sum() == 0, "Trailing pipe found"
assert (df["all_names"].str.len() > 0).all(), "Empty all_names found"

df.to_csv(OUTPUT_PATH, index=False)
print(f"Written {len(df)} entities to {OUTPUT_PATH}")
print(df["entity_type"].value_counts().to_string())

Written 213 entities to ../../output/water_entity_dictionary.csv
entity_type
water_district          64
tribe                   40
state_agency            38
ngo                     30
federal_agency          19
regional_water_board     9
irwm_region              7
irwm_program             6
